# Stage 00 — Project Scope and Source Registry

## Phase 0 — Step 0.1: Jupyter Environment and Project-File Inventory

**Project:** Open Materials Data Pipeline for Hydrogel Bioink Design  
**Primary material system:** Phenol-modified alginate (ALG-Ph) and phenol-modified hyaluronic acid (HA-Ph)  
**Primary source:** [Zenodo record 19602891](https://zenodo.org/records/19602891), planned source version `v0.0.2`  
**Checkpoint:** Environment and file-presence audit; no data extraction or scientific analysis

This notebook records the Python execution context and gives a bounded inventory of the expected project files. The output supports a reproducible handoff: it identifies where the notebook ran, which expected source files were visible, and which parts of the workflow remain outside this checkpoint.

<details>
<summary><strong>Notebook scope and working boundary</strong></summary>

This stage establishes the computational starting point for the project. It checks the active Python environment, identifies the project directory when the expected layout is present, and reports whether the two primary Zenodo source files are available in the designated raw-data folder.

The inventory is intentionally shallow and read-only. It does not open or parse source data, extract ZIP archives, create directories, download files, alter records, or infer anything about the material response. This keeps source acquisition separate from later data validation and analysis, while making the starting state visible in the notebook.

The code uses paths relative to the detected project root rather than a machine-specific absolute path. After reopening the notebook in a fresh kernel, **Run All** reconstructs its variables in order from the imports and path checks below.
</details>

<details>
<summary><strong>Data-science rationale</strong></summary>

Environment provenance matters because package versions and execution context can affect parsing, tabular transformations, plotting, and numerical results. Recording the interpreter and a small set of relevant package versions gives later stages a basis for diagnosing differences between runs; it is not a substitute for a fully pinned environment specification.

The project-root check makes file discovery portable across common launch locations, such as the repository root and its `notebooks/` folder. The inventory then checks named locations and filenames, rather than searching the entire computer. This bounds the audit, avoids accidental use of unrelated files, and produces an explicit missing-file report instead of silently substituting another source.

A file-presence check is a provenance and workflow check only. It does not establish that an archive is intact, that a table conforms to a schema, that records are complete, or that two files are correctly linked. Those claims require separate integrity, schema, and linkage checks in later stages.
</details>

<details>
<summary><strong>Materials-science context</strong></summary>

The Zenodo source describes rheological measurements for ALG-Ph and HA-Ph hydrogel inks, alongside printing metadata and indexed construct images. In this material system, storage modulus (`G′`) and loss modulus (`G″`) are dynamic-mechanical observables associated with the elastic and viscous components of the measured response under the reported test conditions. Viscosity is also condition-dependent; its interpretation depends on the measurement protocol and the source’s recorded variables.

These quantities can inform later comparisons of ink processability and structure–property relationships, but their meaning depends on formulation, sample preparation, test geometry, temperature, strain, angular frequency or shear rate, and the source’s experimental design. The present checkpoint only confirms whether source files are visible. It does not calculate rheological properties, establish a formulation–response relationship, or treat image-index entries as validated sample identities.

Keeping the original source files intact and separating raw data from derived tables preserves provenance. Any later cleaning, unit conversion, feature derivation, or sample linkage should be documented as a transformation with its source fields and assumptions stated explicitly.
</details>


## 1. Source retrieval and expected files

The source record is the [Zenodo dataset page](https://zenodo.org/records/19602891). The checkpoint’s primary source pair is the rheology archive and its image index. The record also lists large image and model archives; those are outside the initial file-presence check and can be staged when a later analysis requires them.

<details>
<summary><strong>Download record and file-selection details</strong></summary>

The record page presents the source files under its **Files** section. The source selection for the first data audit is:

| File | Role in this project | Checkpoint status |
|---|---|---|
| `ALG-Ph_HA-Ph_rheology_data.zip` | Primary archive of rheology data | Expected in the designated raw-data folder |
| `image_index.csv` | Source-provided index for the construct images | Expected in the designated raw-data folder |
| `csv_data_files_generation.zip` | Supporting source-author CSV-generation files | Optional for this checkpoint |
| `images.zip` | Image archive (large) | Deferred until image analysis is in scope |
| `models.zip` | Model archive (large) | Deferred until model use is in scope |

The browser retrieval sequence is: open the record page, confirm the record/version shown on the page, and use the download control beside each selected filename. The expected local destination is `data/raw/zenodo_19602891/`, with the downloaded filenames preserved. The rheology archive and image index remain separate source files; the archive is not expanded during this environment and inventory checkpoint.

The notebook does not perform these downloads. Its file-presence report reflects only files already placed in the expected project location. Source redistribution terms should be checked before raw files are committed to a public repository.
</details>


## 2. Imports and execution context

The following cell records the interpreter, platform, notebook-kernel availability, and versions of commonly used data-science packages. Optional packages are queried through package metadata, so this environment check does not require those packages to be importable.


In [4]:
from importlib import metadata
from pathlib import Path
import platform
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

# Jupyter defines get_ipython in the active kernel namespace. Checking globals
# avoids making IPython a dependency when this cell is run as a plain script.
ipython_getter = globals().get("get_ipython")
try:
    shell = ipython_getter() if callable(ipython_getter) else None
except Exception:
    shell = None

print(f"Interactive shell: {type(shell).__name__ if shell is not None else 'not detected'}")

for package_name in ("ipython", "ipykernel", "numpy", "pandas", "matplotlib", "seaborn"):
    try:
        package_version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        package_version = "not installed"
    print(f"{package_name}: {package_version}")


Python executable: C:\Users\sufer\anaconda3\python.exe
Python version: 3.12.4
Platform: Windows-11-10.0.26200-SP0
Interactive shell: ZMQInteractiveShell
ipython: 8.25.0
ipykernel: 6.28.0
numpy: 2.0.2
pandas: 2.2.2
matplotlib: 3.10.0
seaborn: 0.13.2


## 3. Locate the project directory

The expected repository layout contains a `README.md` and a `notebooks/` directory at the project root. The search checks the current working directory and its parent directories, which supports execution from either the repository root or a notebook subfolder. If those markers are absent, the current working directory is retained as the audit base and the status is reported explicitly.


In [6]:
current_directory = Path.cwd().resolve()
search_locations = (current_directory, *current_directory.parents)
project_root = next(
    (
        location
        for location in search_locations
        if (location / "README.md").is_file()
        and (location / "notebooks").is_dir()
    ),
    None,
)

if project_root is None:
    project_root = current_directory
    project_root_status = "Project markers not detected; using the current working directory as the audit base."
else:
    project_root_status = "Project root identified by README.md and notebooks/."

print(f"Current working directory: {current_directory}")
print(f"Project root used: {project_root}")
print(project_root_status)


Current working directory: C:\Users\sufer\Documents\hydrogel-bioink
Project root used: C:\Users\sufer\Documents\hydrogel-bioink
Project markers not detected; using the current working directory as the audit base.


## 4. Inventory expected project locations

The inventory reports immediate entries in the project root and the designated data and notebook locations. It is non-recursive: archive contents and nested folders are not inspected. This keeps the output small and makes clear that presence on disk is the only property assessed here.


In [8]:
inventory_paths = {
    "project root": project_root,
    "data": project_root / "data",
    "raw data": project_root / "data" / "raw",
    "Zenodo source folder": project_root / "data" / "raw" / "zenodo_19602891",
    "metadata": project_root / "data" / "metadata",
    "notebooks": project_root / "notebooks",
}

for label, directory in inventory_paths.items():
    print(f"\n{label}: {directory}")
    if not directory.is_dir():
        print("  [directory not present]")
        continue

    entries = sorted(directory.iterdir(), key=lambda path: (not path.is_dir(), path.name.casefold()))
    if not entries:
        print("  [empty]")
    else:
        for entry in entries:
            kind = "directory" if entry.is_dir() else "file"
            print(f"  [{kind}] {entry.name}")



project root: C:\Users\sufer\Documents\hydrogel-bioink
  [directory] .ipynb_checkpoints
  [directory] data
  [file] 00_scope_and_source_registry.ipynb

data: C:\Users\sufer\Documents\hydrogel-bioink\data
  [directory] raw

raw data: C:\Users\sufer\Documents\hydrogel-bioink\data\raw
  [directory] zenodo_19602891

Zenodo source folder: C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891
  [file] ALG-Ph_HA-Ph_rheology_data.zip
  [file] image_index.csv

metadata: C:\Users\sufer\Documents\hydrogel-bioink\data\metadata
  [directory not present]

notebooks: C:\Users\sufer\Documents\hydrogel-bioink\notebooks
  [directory not present]


## 5. Primary source-file availability

The final cell evaluates whether each selected source file is present in `data/raw/zenodo_19602891/`. A missing result means the file is not at that expected location; it does not imply that the file is unavailable elsewhere or that the source record is incomplete. No archive integrity or content checks are performed at this stage.


In [10]:
zenodo_data_directory = project_root / "data" / "raw" / "zenodo_19602891"
expected_source_files = (
    "ALG-Ph_HA-Ph_rheology_data.zip",
    "image_index.csv",
)

source_file_status = {
    filename: (zenodo_data_directory / filename).is_file()
    for filename in expected_source_files
}

print(f"Expected source directory: {zenodo_data_directory}")
for filename, present in source_file_status.items():
    status = "present" if present else "not found"
    print(f"{filename}: {status}")

available_count = sum(source_file_status.values())
print(f"\nPrimary source files present: {available_count}/{len(expected_source_files)}")


Expected source directory: C:\Users\sufer\Documents\hydrogel-bioink\data\raw\zenodo_19602891
ALG-Ph_HA-Ph_rheology_data.zip: present
image_index.csv: present

Primary source files present: 2/2


## 6. Checkpoint interpretation and limitations

This checkpoint establishes whether the notebook can describe its execution context and whether the selected source files are visible in the expected location. A successful environment check and file-presence check provide a reproducible starting record; they do not establish data validity or scientific conclusions.

The next data-handling stage can verify archive integrity, inspect archive members without overwriting the original, and assess the image index schema before any structured extraction or sample-linkage work. Later materials analysis should preserve source-reported experimental conditions and distinguish observed measurements from derived quantities.

**Checkpoint result:** environment and expected-file status are reported by the executed cells above. No download, extraction, or data transformation is carried out by this notebook.
